# Frame-Level Anomaly Detection Evaluation

Evaluate temporal anomaly localization on UCF-Crime test set.

**Frame-Level vs Clip-Level:**
- Clip-level: Treats 16-frame segments independently
- Frame-level: Measures precise temporal localization (when anomalies start/end)
- Standard benchmark metric for surveillance anomaly detection

In [ ]:
import sys
import torch
import numpy as np
import pickle
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
from IPython.display import display, Image

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.models.r3d import create_r3d_classifier
from src.datasets.transforms import RGBVideoTransform
from src.utils.frame_level_evaluation import evaluate_frame_level
from src.utils.training_utils import load_checkpoint
from src.visualization.plots import plot_roc_curve, plot_best_worst_videos

%matplotlib inline


KeyboardInterrupt: 

## 1. Select Checkpoint

In [ ]:
def find_checkpoints(base_dir="/work3/s225224/ucf-crime/checkpoints"):
    base_path = Path(base_dir)
    return sorted(
        [
            d / "best_model.pth"
            for d in base_path.glob("r3d_*")
            if (d / "best_model.pth").exists()
        ]
    )


checkpoints = find_checkpoints()
for i, ckpt in enumerate(checkpoints):
    print(f"[{i}] {ckpt.parent.name}")

# Select checkpoint
checkpoint_path = checkpoints[0]  # Change index here
print(f"\nSelected: {checkpoint_path.parent.name}")

## 2. Run Evaluation (~10 min on GPU)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results_dir = (
    Path("/work3/s225224/ucf-crime/experiments/frame_level")
    / f"{checkpoint_path.parent.name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)
results_dir.mkdir(parents=True, exist_ok=True)

# Load model
model = create_r3d_classifier(
    num_classes=2, pretrained=False, freeze_backbone=True, dropout=0.5
)
load_checkpoint(checkpoint_path, model, device=device)
model.to(device).eval()

# Evaluate
transform = RGBVideoTransform(mode="val", crop_size=112, resize_size=128)
metrics, video_results = evaluate_frame_level(
    model,
    "/work3/s225224/ucf-crime/data/Test",
    "/work3/s225224/ucf-crime/data/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
    transform,
    device,
    clip_len=16,
    stride=16,
    sigma=5,
)

print(f"Frame-Level AUC: {metrics.frame_auc:.4f}")
print(f"Total Frames: {metrics.num_frames:,}")
print(f"Total Videos: {metrics.num_videos}")

## 3. Generate Plots

In [ ]:
plot_dir = results_dir / "plots"
plot_dir.mkdir(exist_ok=True)

# Save data
np.savez(
    results_dir / "raw_data.npz",
    fpr=metrics.fpr,
    tpr=metrics.tpr,
    frame_auc=metrics.frame_auc,
)
with open(results_dir / "video_results.pkl", "wb") as f:
    pickle.dump(video_results, f)

# Plot
plot_roc_curve(metrics.fpr, metrics.tpr, metrics.frame_auc, plot_dir / "roc_curve.png")
plot_best_worst_videos(video_results, {}, plot_dir, top_n=5)

print(f"Saved to: {plot_dir}")

## 4. Visualize Results

### ROC Curve

In [ ]:
display(Image(filename=str(plot_dir / "roc_curve.png")))

### Best Videos (High AUC)

In [ ]:
for img in sorted(plot_dir.glob("best_*.png")):
    display(Image(filename=str(img)))

## 5. Statistics

In [ ]:
if video_results:
    aucs = [v[1] for v in video_results]
    print(f"Mean Video AUC: {np.mean(aucs):.4f}")
    print(f"Median: {np.median(aucs):.4f}")
    print(f"Videos with AUC > 0.8: {sum(a > 0.8 for a in aucs)}/{len(aucs)}")

    plt.figure(figsize=(10, 6))
    plt.hist(aucs, bins=30, edgecolor="black", alpha=0.7)
    plt.axvline(
        np.mean(aucs), color="r", linestyle="--", label=f"Mean: {np.mean(aucs):.3f}"
    )
    plt.xlabel("Video-Level AUC")
    plt.ylabel("Count")
    plt.title("Distribution of Per-Video AUC")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Summary

- **Frame-Level AUC**: Primary metric for temporal localization
- **Good performance**: AUC > 0.75
- **State-of-the-art**: ~0.82 (Sultani et al. 2018)

Results saved to: `{results_dir}`